Install packages

In [1]:
# import sys

# !{sys.executable} -m pip install -U memory_profiler

In [2]:
import graphblas as gb
from graphblas import Matrix, Vector, dtypes, unary, binary, monoid, semiring
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time
import warnings
import urllib
import ssl
import gzip
import tracemalloc
import random

from pathlib import Path
from functools import partial
from collections import OrderedDict
from memory_profiler import memory_usage

In [3]:
def create_stochastic_matrix(A, use_fp32=False):
    """
    Преобразует бинарную матрицу смежности в стохастическую матрицу M.
    Гарантирует, что каждая строка i суммируется в 1 (если d_out[i] > 0).
    """
    if use_fp32:
        dtype_for_obj = dtypes.FP32
    else:
        dtype_for_obj = dtypes.FP64
        
    n = A.nrows
    d_out = A.reduce_rowwise(monoid.plus).new()

    inv_d = Vector(dtype_for_obj, n)
    mask = (d_out > 0).new()
    inv_d(mask=mask.S) << binary.truediv(1.0, d_out)

    # Создаём диагональную матрицу D[i,i] = 1/d_out[i]
    D = Matrix.diag(inv_d)
    # Умножение D @ A масштабирует СТРОКИ: M[i,j] = D[i,i] * A[i,j]
    M = D @ A

    return M, d_out

In [4]:
def pagerank_basic(
    A, 
    damping=0.85, 
    max_iter=100, 
    tol=1e-6, 
    verbose=False, 
    use_fp32= False, 
    use_chunks=False, 
    chunks_num: int = 2
):
    if use_fp32:
        dtype_for_obj = dtypes.FP32
    else:
        dtype_for_obj = dtypes.FP64
    
    n = A.nrows
    # print(n)
    teleport = (1 - damping) / n
    M, d_out = create_stochastic_matrix(A)

    r = Vector(dtype_for_obj, n)
    r[:] << 1.0 / n
    t = Vector(dtype_for_obj, n)
    w = Vector(dtype_for_obj, n)

    history = {'rdiff': [], 'time_per_iter': []}

    if use_chunks:
        chunks = M.T.new().ss.split([chunks_num, None])
    for iteration in range(max_iter):
        start_time = time.time()
        t << r
        if use_chunks:
            result_tiles = []
            for chunk in chunks:
                result_tiles.append(chunk[0].mxv(t, semiring.plus_times).new())
            # print(result_tiles)
            w << gb.ss.concat(result_tiles)
        else:
            w << M.T.mxv(t, semiring.plus_times)
        r << binary.times(damping, w)
        r(binary.plus)[:] << teleport

        diff = t.ewise_mult(r, binary.minus).apply(unary.abs)
        rdiff = diff.reduce(monoid.plus).new().value

        iter_time = time.time() - start_time
        history['rdiff'].append(rdiff)
        history['time_per_iter'].append(iter_time)

        if verbose and iteration % 10 == 0:
            print(f"Iter {iteration:3d}: rdiff={rdiff:.2e}, time={iter_time*1000:.2f}ms")

        if rdiff < tol:
            if verbose: print(f"✓ Сходимость достигнута на итерации {iteration+1}")
            break

    return r, history

In [5]:
# Создаём небольшой тестовый граф для проверки
A_test = Matrix.from_coo(
    [0, 0, 1, 2],          # rows
    [1, 2, 3, 3],          # columns
    [1]*4,                          # values
    dtype=dtypes.UINT8,
    nrows=4,
    ncols=4,
)

print("Тестовый граф (матрица смежности):")
print(A_test)

Тестовый граф (матрица смежности):
"M_0"      nvals  nrows  ncols  dtype         format
gb.Matrix      4      4      4  UINT8  bitmapr (iso)
----------------------------------------------------
  0  1  2  3
0    1  1   
1          1
2          1
3           


## Function to download graphs from SNAP

In [6]:
def load_snap_dataset(dataset_name, cache_dir='./snap_datasets', use_fp32=False):
    """
    Загружает граф из коллекции SNAP (Stanford Network Analysis Project)

    Parameters:
    -----------
    dataset_name : str - имя датасета (например, 'ego-facebook', 'wiki-Vote')
    cache_dir : str - директория для кэширования загруженных файлов

    Returns:
    --------
    A : gb.Matrix - матрица смежности в формате GraphBLAS
    meta : dict - метаданные графа (количество узлов, рёбер, плотность)
    """
    # Каталог для кэширования датасетов
    cache_path = Path(cache_dir)
    cache_path.mkdir(exist_ok=True)

    # URL для популярных датасетов SNAP
    snap_urls = {
        'ego-facebook': 'https://snap.stanford.edu/data/facebook_combined.txt.gz',
        'wiki-Vote': 'https://snap.stanford.edu/data/wiki-Vote.txt.gz',
        'web-Stanford': 'https://snap.stanford.edu/data/web-Stanford.txt.gz',
        'soc-epinions1': 'https://snap.stanford.edu/data/soc-Epinions1.txt.gz',
    }

    if dataset_name not in snap_urls:
        print(f"Датасет '{dataset_name}' не найден в предустановленных.")
        print(f"Доступные датасеты: {list(snap_urls.keys())}")
        return None, None

    # Скачивание и распаковка файла
    url = snap_urls[dataset_name]
    filename = url.split('/')[-1]
    filepath = cache_path / filename
    txt_path = cache_path / filename.replace('.gz', '')

    if not txt_path.exists():
        print(f"Загрузка {dataset_name}...")
        if not filepath.exists():
            urllib.request.urlretrieve(url, filepath)
        with gzip.open(filepath, 'rt') as f_in:
            with open(txt_path, 'w') as f_out:
                for line in f_in:
                    if not line.startswith('#') and line.strip():
                        f_out.write(line)

    # Чтение графа через NetworkX для удобства парсинга
    # print(f"Чтение графа из {txt_path.name}...")
    G = nx.read_edgelist(txt_path, create_using=nx.DiGraph())

    # Преобразование в формат GraphBLAS
    # print("Конвертация в формат GraphBLAS...")
    tracemalloc.start()
    if use_fp32:
        A = gb.io.from_networkx(G, dtype=dtypes.FP32)
    else:
        A = gb.io.from_networkx(G, dtype=dtypes.UINT8)
    _, peak_size = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # Сбор метаданных
    meta = {
        'name': dataset_name,
        'n_nodes': A.nrows,
        'n_edges': A.nvals,
        'density': A.nvals / (A.nrows * A.ncols) if A.nrows > 0 else 0,
        'source': 'SNAP'
    }

    # print(f"Загружено: {meta['n_nodes']:,} узлов, {meta['n_edges']:,} рёбер")
    # print(f"   Плотность графа: {meta['density']:.2e}")

    return A, meta, peak_size

In [7]:
ssl._create_default_https_context = ssl._create_unverified_context

Загружаем web-Stanford

In [8]:
web_stanford_graph, _, _ = load_snap_dataset("web-Stanford")

In [9]:
# r, _ = pagerank_basic(web_stanford_graph)

Запускаем разные варианты PageRank на графе web-Stanford

In [10]:
algorithms_options = OrderedDict({
    "base": partial(pagerank_basic, use_fp32=False, use_chunks=False),
    "fp32": partial(pagerank_basic, use_fp32=True, use_chunks=False),
    # "2_chunks": partial(pagerank_basic, use_fp32=False, use_chunks=True, chunks_num=2),
    # "10_chunks": partial(pagerank_basic, use_fp32=False, use_chunks=True, chunks_num=10),
    "100_chunks": partial(pagerank_basic, use_fp32=False, use_chunks=True, chunks_num=100),
    "1000_chunks": partial(pagerank_basic, use_fp32=False, use_chunks=True, chunks_num=1000),
    "10000_chunks": partial(pagerank_basic, use_fp32=False, use_chunks=True, chunks_num=10000),
    "100000_chunks": partial(pagerank_basic, use_fp32=False, use_chunks=True, chunks_num=100000),
    "1000000_chunks": partial(pagerank_basic, use_fp32=False, use_chunks=True, chunks_num=1000000),
    "100_chunks_fp32": partial(pagerank_basic, use_fp32=True, use_chunks=True, chunks_num=100),
    "1000_chunks_fp32": partial(pagerank_basic, use_fp32=True, use_chunks=True, chunks_num=1000),
    "10000_chunks_fp32": partial(pagerank_basic, use_fp32=True, use_chunks=True, chunks_num=10000),
    "100000_chunks_fp32": partial(pagerank_basic, use_fp32=True, use_chunks=True, chunks_num=100000),
    "1000000_chunks_fp32": partial(pagerank_basic, use_fp32=True, use_chunks=True, chunks_num=1000000),
})

results = dict(
    algo_name=[],
    time=[],
    memory_usage_mb=[],
    num_iterations_to_converge=[],
    max_diff_rang_base_solution=[],
)

base_rang = None
for algorithm_name, algorithm_func in algorithms_options.items():
    if "fp32" in algorithm_name:
        web_stanford_graph, _, peak_size_initial_graph = load_snap_dataset("web-Stanford", use_fp32=True)
    else:
        web_stanford_graph, _, peak_size_initial_graph = load_snap_dataset("web-Stanford", use_fp32=False)
        
    # print(algorithm_name)
    start_time = time.perf_counter_ns()
    tracemalloc.start()
    current_r, current_history = algorithm_func(web_stanford_graph)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    total_time = (time.perf_counter_ns() - start_time)
    total_time /= 1e9

    # Look for the memory usage of the current algorithm
    # memory_usage_current_algo = memory_usage(lambda: algorithm_func(web_stanford_graph))
    results["memory_usage_mb"].append((peak + peak_size_initial_graph) / (1024. ** 2))

    if algorithm_name == "base":
        base_rang = current_r.to_dense()
        results["max_diff_rang_base_solution"].append(0.0)
    else:
        current_rang = current_r.to_dense()
        curr_max_diff = max(np.absolute(base_rang - current_rang))
        results["max_diff_rang_base_solution"].append(curr_max_diff)

    results["algo_name"].append(algorithm_name)
    results["num_iterations_to_converge"].append(len(current_history["time_per_iter"]))
    # results["time"].append(total_time)
    results["time"].append(sum(current_history["time_per_iter"]))

In [11]:
results_df = pd.DataFrame(results)
results_df

,algo_name,time,memory_usage_mb,num_iterations_to_converge,max_diff_rang_base_solution
0,base,0.434884,694.467077,63,0.000000e+00
1,fp32,1.023610,596.991897,63,8.797233e-10
2,100_chunks,9.443712,725.095683,63,1.606354e-15
3,1000_chunks,1.006469,632.455398,63,1.606354e-15
4,10000_chunks,0.313090,595.953909,63,1.606354e-15
5,100000_chunks,0.178951,709.348197,63,1.606354e-15
6,1000000_chunks,0.142992,837.440262,63,1.606354e-15
7,100_chunks_fp32,9.843240,792.155881,63,8.797233e-10
8,1000_chunks_fp32,1.380611,570.595574,63,8.797233e-10
9,10000_chunks_fp32,0.574422,568.442412,63,8.797233e-10


- Лучше всего показал себя вариант при разбиении на 10000 chunks и использовании fp32: 568.44mb на все против 694.46 mb в случае бейзлайна.

- Всем вариантам алгоритмов потребовалось 63 итерации для того, чтобы сойтись.

-  Быстрее всего отработал алгоритм при разбиении на 10**6 chunks и fp64 (0.142 сек против 0.43 в случае бейзлайна). При этом при увеличении кол-ва chunks, на которые мы разбиваем матрицу M время работы алгоритма падает, но при этом растет потребление памяти (скорее всего из-за материализации чанков и new).

-  При использовании fp32 максимальная разница в рангах по сравнению с бейзлайном (base) не превышает 8.797233 * e-10. В случае использования fp64 макс. разница сильно меньше: 1.66 * e-15.

In [12]:
def add_edges_to_matrix(A_old, new_edges):
    # dtype_for_obj = dtypes.FP64  # dtypes.FP64
    dtype_for_obj = dtypes.FP64
        
    src_lst = [el[0] for el in new_edges]
    dst_lst = [el[1] for el in new_edges]
    
    delta_A = Matrix.from_coo(
        src_lst,
        dst_lst,
        [1] * len(src_lst),
        dtype=dtype_for_obj,
        nrows=A_old.nrows,
        ncols=A_old.ncols,
    )

    return A_old.ewise_add(delta_A, op="plus")
    

# Формула обновления (линеаризация):
#   Δr ≈ β * M^T * Δr + β * ΔM^T * r_old
def incremental_pagerank(
    A_old, 
    r_old,
    new_edges,
    damping=0.85,
    max_iter=50,
    tol=1e-6,
    dtype_for_obj=dtypes.FP64,
    verbose=False
):
    """
    Parameters:
    -----------
    A_old : gb.Matrix - исходная матрица смежности (бинарная)
    r_old : gb.Vector - исходный вектор рангов (стационарное решение)
    new_edges : list of tuple - новые рёбра [(src, dst), ...]
    damping : float - коэффициент демпфирования β  
    max_iter : int - число итераций для уточнения обновления
    
    Returns:
    --------
    r_new : gb.Vector - обновлённый вектор рангов (приближённый)
    """
    # Создаем новую матрицу A_new, используя новые edges
    A_new = add_edges_to_matrix(A_old, new_edges)

    # Создаем stochastic матрицы
    M_old, d_out_old = create_stochastic_matrix(A_old)
    M_new, d_out_new = create_stochastic_matrix(A_new)

    # Считаем delta M
    deltaM = Matrix(dtype_for_obj, M_new.nrows, M_old.ncols)
    deltaM << M_new.ewise_add(M_old, op=binary.minus)

    # Заводим вектор для delta r - изначально инициализиация 0
    delta_r = Vector(dtype_for_obj, size=r_old.size)
    delta_r << 0.0
    
    previous_delta_r = Vector(dtype_for_obj, size=r_old.size)
    previous_delta_r << 0

    # Вектора для слагаемых
    first_part = Vector(dtype_for_obj, size=r_old.size)
    
    # Второе слагаемое можно посчитать до цикла
    second_part = Vector(dtype_for_obj, size=r_old.size)
    second_part << deltaM.T.mxv(r_old)

    cnt_iter = 0
    for iteration in range(max_iter):
        # Записываем в previous_delta_r текущее значение r
        previous_delta_r << delta_r

        # Считаем новый delta r
        first_part << M_old.T.mxv(previous_delta_r)
        delta_r << binary.times(damping, first_part).ewise_add(binary.times(damping, second_part), op=binary.plus)

        # Считаем изменение на текущем шаге
        diff = delta_r.ewise_mult(previous_delta_r, binary.minus).apply(unary.abs)  # ewise_add
        rdiff = diff.reduce(monoid.plus).new().value

        # Если изменение слишком мало, то заканчиваем итерироваться
        if rdiff < tol:
            if verbose: 
                print(f"Сходимость достигнута на итерации {iteration + 1}")
            break
        cnt_iter += 1

    # Возвращаем r_new
    r_new = r_old.ewise_add(delta_r, op=binary.plus).new()
    return r_new, A_new, cnt_iter

Загружаем web-Stanford.

In [13]:
web_stanford_graph, _, _ = load_snap_dataset("web-Stanford", use_fp32=False)

In [14]:
def create_new_edges(num_new_edges, A_old):
    random_added_pairs = set()
    new_edges = []
    max_num_iter = 10000

    def check_edge_exists(A_old, i, j):
        return A_old[i, j].value == 1

    for _ in range(num_new_edges):
        i, j = random.randint(0, A_old.ncols), random.randint(0, A_old.ncols)
        current_iter = 1
        while ((i, j) in random_added_pairs) or ((j, i) in random_added_pairs) or check_edge_exists(A_old, i, j):
            if current_iter > max_num_iter:
                break
            i, j = random.randint(0, A_old.ncols), random.randint(0, A_old.ncols)
            current_inter += 1
        random_added_pairs.add((i, j))
        new_edges.append((i, j))

    return new_edges

In [15]:
new_edges = create_new_edges(100, web_stanford_graph)

In [16]:
# new_edges

In [17]:
r_web_stanford_base, _ = pagerank_basic(web_stanford_graph, use_fp32=False)

Запускаем incremental PageRank

In [18]:
start = time.perf_counter_ns()
r_web_stanford_new_incremental, _, total_iterations = incremental_pagerank(
    web_stanford_graph,
    r_web_stanford_base,
    new_edges,
)
incremental_time = (time.perf_counter_ns() - start) / 1e9

Запускаем обычный PageRank basic на графе с новыми вершинами

In [19]:
start = time.perf_counter_ns()
web_stanford_new = add_edges_to_matrix(
    web_stanford_graph,
    new_edges
)
r_web_stanford_new_basic_pagerank, basic_option_history = pagerank_basic(web_stanford_new, use_fp32=False)
basic_pagerank_time = (time.perf_counter_ns() - start) / 1e9

In [20]:
print(f"[Метод с incremental page rank] Время в сек: {incremental_time:.4f}")
print(f"[Метод с basic page rank] Время в сек: {basic_pagerank_time:.4f}")

[Метод с incremental page rank] Время в сек: 0.1939
[Метод с basic page rank] Время в сек: 0.4072


Обновление page rank с помощью incremental метода быстрее, чем метод: изменение матрицы и обычный пересчет через basic pagerank.

In [22]:
print(f"[Метод с incremental page rank] кол-во итераций для сходимости: {total_iterations}")
num_iterations_basic = len(basic_option_history["time_per_iter"])
print(f"[Метод с basic page rank] кол-во итераций для сходимости: {num_iterations_basic}")

[Метод с incremental page rank] кол-во итераций для сходимости: 24
[Метод с basic page rank] кол-во итераций для сходимости: 63


In [23]:
max_difference = max(np.absolute(r_web_stanford_new_basic_pagerank.to_dense() - r_web_stanford_new_incremental.to_dense()))
print(f"Максимальная разница в рангах: incremental method vs base method = {max_difference}")

Максимальная разница в рангах: incremental method vs base method = 9.776705729390685e-07


In [24]:
# max(np.absolute(r_web_stanford_new_basic_pagerank.to_dense() - r_web_stanford_base.to_dense()))

**Вывод:** для обновления PageRANK при добавлении новых edges выгодно по времени использовать incremental method, по сравнению с basic page rank методом